# PDE Generator: Architecture Report

This notebook documents the composable symbolic framework for deriving
depth-averaged equations from the 3D Incompressible Navier-Stokes (INS).

**Core idea:** the user drives every step. No hidden assumptions.

| Building block | What it does |
|---------------|-------------|
| `Expression` | Symbolic wrapper with `.terms`, `[i]`, `.project()`, `.ibp()`, `.apply()` |
| `FullINS` | All 4 equations (continuity, x/y/z momentum) with full stress tensor |
| `materials` | Constitutive model library (Newtonian, inviscid, ...) |
| `assumptions` | Physical conditions (kinematic BCs, hydrostatic pressure, ...) |

In [ ]:
import sympy as sp
from sympy import Symbol, Function, Derivative, S, Rational, Integral
sp.init_printing()

---
## 1. The Full INS Equations

In [ ]:
from zoomy_core.model.models.ins_generator import (
    FullINS, Expression, integrate_by_parts,
    materials, assumptions,
)

ins = FullINS(dimension=1)
print(f"Fields: u={ins.u}, w={ins.w}, p={ins.p}")
print(f"Stress tensor: tau_xz={ins.tau['xz']}, tau_zz={ins.tau['zz']}, ...")
print(f"Bathymetry: b={ins.b}, H={ins.H}, eta={ins.eta}")

### All four equations are available

In [ ]:
for eq in ins.equations:
    print(f"\n{eq.name} ({len(eq)} terms):")
    display(eq.expr)

### Browsing terms

Each equation supports `[i]` indexing and iteration:

In [ ]:
xm = ins.x_momentum
print(f"X-momentum has {len(xm)} terms:\n")
for i, term in enumerate(xm):
    display(sp.Eq(Symbol(f"T_{i}"), term.expr))

### Dropping terms manually

No magic switches. The user removes what they want:

In [ ]:
# Drop z-momentum inertia to get hydrostatic balance
zm = ins.z_momentum
print("Full z-momentum:")
display(zm.expr)

# Identify and remove inertia terms (those with Derivative(w, t), Derivative(w*u, x), etc.)
# The user inspects and decides which terms to keep
zm_no_inertia = Expression(S.Zero, "z_momentum_hydrostatic")
for term in zm:
    if not term.has(ins.w) or term.has(ins.tau["zz"]) or term.has(ins.tau["zx"]):
        zm_no_inertia = zm_no_inertia + term
    elif term.expr == ins.g:
        zm_no_inertia = zm_no_inertia + term
    elif term.has(ins.p):
        zm_no_inertia = zm_no_inertia + term

print("\nHydrostatic balance (manually dropped inertia):")
display(zm_no_inertia.expr)

---
## 2. Material Models

The stress tensor $\tau_{ij}$ is abstract until the user chooses a constitutive model.
Models are applied via `.apply()` — on the full equation or on individual terms.

In [ ]:
# Newtonian: tau_ij = mu * (du_i/dx_j + du_j/dx_i)
newton = materials.newtonian(ins)
print(f"Newtonian model: {newton}")
print(f"  nu = {newton.nu}")

Apply to full x-momentum:

In [ ]:
xm_newton = ins.x_momentum.apply(newton)
print("X-momentum with Newtonian stress:")
display(xm_newton.simplify().expr)

Apply to a single term:

In [ ]:
stress_term = ins.x_momentum[2]
print(f"Original stress term: {stress_term.expr}")
result = stress_term.apply(newton)
print(f"After Newtonian:      {result.simplify().expr}")

**Inviscid:** set all $\tau_{ij} = 0$:

In [ ]:
xm_inviscid = ins.x_momentum.apply(materials.inviscid(ins))
print("Inviscid x-momentum:")
display(xm_inviscid.expr)

**Custom material:** just provide a substitution dict:

```python
# Power-law fluid example
my_model = {
    ins.tau["xz"]: K * abs(Derivative(ins.u, ins.z))**(n-1) * Derivative(ins.u, ins.z),
    ...
}
xm_custom = ins.x_momentum.apply(my_model)
```

---
## 3. Assumptions Library

Physical conditions applied via `.apply()`. Chainable.

### Hydrostatic pressure

Substitutes $p = p_{atm} + \rho g (\eta - z)$:

In [ ]:
hydro = assumptions.hydrostatic_pressure(ins)
xm_hydro = ins.x_momentum.apply(hydro)
print("X-momentum with hydrostatic pressure:")
display(xm_hydro.simplify().expr)

### Kinematic boundary conditions

- Bottom: $w|_{z=b} = \partial_t b + u_b \,\partial_x b$
- Surface: $w|_{z=\eta} = \partial_t \eta + u_s \,\partial_x \eta$

In [ ]:
kbc_bot = assumptions.kinematic_bc_bottom(ins)
kbc_top = assumptions.kinematic_bc_surface(ins)

print("Kinematic BC (bottom):")
for old, new in kbc_bot.subs_map.items():
    display(sp.Eq(old, new))

print("\nKinematic BC (surface):")
for old, new in kbc_top.subs_map.items():
    display(sp.Eq(old, new))

### Chaining: hydrostatic + inviscid in one call

In [ ]:
xm_simple = ins.x_momentum.apply(
    assumptions.hydrostatic_pressure(ins),
    materials.inviscid(ins),
)
print("Hydrostatic + inviscid x-momentum:")
display(xm_simple.simplify().expr)

---
## 4. Projection and Integration by Parts

### Basic projection

`.project(test, var, domain)` works on ANY Expression:

In [ ]:
zeta = ins.zeta
phi_0 = S.One  # constant test function

cont = ins.continuity
projected = cont.project(phi_0, ins.z, domain=(ins.b, ins.eta))
print("Projected continuity:")
display(projected.expr)

### Integration by parts

For the $\partial_z(uw)$ term, explicitly apply IBP:

In [ ]:
duwdz = ins.x_momentum[4]  # d(uw)/dz term
print(f"Term: {duwdz.expr}")

ibp = duwdz.ibp(var=ins.z, test_weight=phi_0, domain=(ins.b, ins.eta))
print(f"\nIBP result:")
print(f"  integrate:       {ibp.integrate.expr}")
print(f"  boundary_upper:  {ibp.boundary_upper.expr}")
print(f"  boundary_lower:  {ibp.boundary_lower.expr}")

### Applying BCs to boundary terms

The user decides which BCs go where:

In [ ]:
ibp_with_bcs = ibp.apply_bcs(
    bc_lower=kbc_bot.subs_map,
    bc_upper=kbc_top.subs_map,
)
print("After kinematic BCs:")
print(f"  upper: {ibp_with_bcs.boundary_upper.expr}")
print(f"  lower: {ibp_with_bcs.boundary_lower.expr}")

# Assemble into single expression
assembled = ibp_with_bcs.assemble()
print(f"\nAssembled: {assembled.expr}")

### Stress boundary conditions

The user provides stress BCs naturally:

In [ ]:
tau_b = Symbol("tau_b")
tau_s = Symbol("tau_s")

stress_bc_bottom = {ins.tau["xz"].subs(ins.z, ins.b): tau_b}
stress_bc_surface = {ins.tau["xz"].subs(ins.z, ins.eta): tau_s}

print("Stress BC (bottom):")
for old, new in stress_bc_bottom.items():
    display(sp.Eq(old, new))

# Apply IBP to the stress divergence term, then apply stress BCs
stress_term = ins.x_momentum[2]  # -1/rho * d(tau_xz)/dz
print(f"\nStress term: {stress_term.expr}")

stress_ibp = stress_term.ibp(var=ins.z, test_weight=phi_0, domain=(ins.b, ins.eta))
stress_with_bcs = stress_ibp.apply_bcs(
    bc_lower=stress_bc_bottom,
    bc_upper=stress_bc_surface,
)
print(f"\nAfter stress BCs:")
print(f"  integrate:  {stress_with_bcs.integrate.expr}")
print(f"  upper: {stress_with_bcs.boundary_upper.expr}")
print(f"  lower: {stress_with_bcs.boundary_lower.expr}")

---
## 5. What this enables

### SWE derivation (user workflow)
```python
ins = FullINS(dimension=1)
cont = ins.continuity
xmom = ins.x_momentum

# Apply hydrostatic + Newtonian
xmom = xmom.apply(assumptions.hydrostatic_pressure(ins), materials.newtonian(ins))

# Project each term
for term in xmom:
    if has_z_derivative(term):
        result = term.ibp(z, phi_k, (b, eta))
        result = result.apply_bcs(...)
    else:
        result = term.project(phi_k, z, (b, eta))
```

### VAM (keep all stress, no hydrostatic assumption)
```python
xmom = ins.x_momentum.apply(materials.newtonian(ins))  # keep full pressure
zmom = ins.z_momentum.apply(materials.newtonian(ins))   # keep full z-inertia
```

### Green-Naghdi (manually drop z-inertia, keep non-hydrostatic pressure)
```python
zmom = ins.z_momentum
# Remove temporal + advection terms, keep pressure + gravity + stress
zmom_constraint = ...  # user selects terms to keep
# Solve for p_nh from the constraint, feed into x-momentum
```

### Energy / turbulence (future)
```python
energy_eq = Expression(Derivative(E, t) + ..., name="energy")
projected = energy_eq.project(phi_k, zeta, (0, 1))
# .project() works on ANY Expression
```

---
## 6. What is still missing

1. **Coordinate transformation** ($z \to \zeta$) — currently manual
2. **Ansatz substitution** — `LayeredAnsatz` from Phase 2 needs integration with `.apply()`
3. **Numerical integration per term** — `.project(..., numerical=True)` exists but untested with real models
4. **Completeness check** — verify all stress BCs are specified and system is closed
5. **Assembly into Model class** — extract flux/source from projected terms
6. **Evolution vs constraint** — tagging z-momentum as constraint when hydrostatic